### Import Library Utama

- **Library Sistem & Utilitas**  
  - `import shutil, os` → Manajemen file & direktori.  
  - `import json` → Membaca/menyimpan konfigurasi dalam format JSON.

- **Library Numerik & Dataframe**  
  - `import numpy as np` → Operasi numerik & array.  
  - `import pandas as pd` → Manipulasi data tabular.

- **Library Preprocessing & Evaluasi**  
  - `from sklearn.preprocessing import StandardScaler, LabelEncoder`  
    → Normalisasi fitur numerik & encoding label target.  
  - `from sklearn.metrics import classification_report, confusion_matrix`  
    → Evaluasi performa model klasifikasi.

- **Library Deep Learning (TensorFlow/Keras)**  
  - `import tensorflow as tf` → Framework utama deep learning.  
  - `from tensorflow.keras.models import load_model, Model`  
    → Memuat model tersimpan & membangun arsitektur baru.  
  - `from tensorflow.keras.preprocessing.image import ImageDataGenerator`  
    → Preprocessing & augmentasi data citra.  
  - `from tensorflow.keras.layers import Input, Lambda, Average`  
    → Layer input, operasi custom (Lambda), dan ensemble (Average).

---

Bagian ini menyiapkan semua library yang dibutuhkan untuk pipeline:  
- **Manipulasi data** (NumPy, Pandas).  
- **Preprocessing & evaluasi** (scikit-learn).  
- **Deep learning** (TensorFlow/Keras).  
- **Manajemen file** (os, shutil, json).  


In [ ]:
import shutil
import os
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Lambda, Average
from tensorflow.keras.models import Model

### Parameter Utama Model

- **SEQ_LEN = 24**  
  → Panjang urutan input untuk model forecasting.  
  → Artinya model menggunakan **24 jam data historis** sebagai jendela input.

- **HORIZON = 1**  
  → Horizon prediksi = 1 langkah ke depan.  
  → Model memprediksi **kondisi cuaca pada jam berikutnya**.

- **TARGET_COLUMN = "cuaca"**  
  → Kolom target yang akan diprediksi.  
  → Berisi label kelas cuaca (misalnya: Cerah, Berawan, Hujan).


In [2]:
SEQ_LEN = 24
HORIZON = 1
TARGET_COLUMN = "cuaca" 

### Fungsi Utilitas untuk Preprocessing & Feature Engineering

- **make_windows(data, labels, seq_len, horizon)**  
  → Membentuk dataset sekuensial untuk model forecasting.  
  - Input: array fitur (`data`), label (`labels`), panjang urutan (`seq_len`), horizon prediksi (`horizon`).  
  - Proses: Sliding window sepanjang `seq_len`, label diambil pada posisi `seq_len + horizon - 1`.  
  - Output: `X` (array sekuensial), `y` (label target).  

- **load_scaler_from_json(path)**  
  → Memuat kembali objek `StandardScaler` dari file JSON.  
  - Membaca parameter `mean` dan `std` dari file.  
  - Menginisialisasi ulang atribut `mean_`, `scale_`, `var_`, dan `n_features_in_`.  
  - Output: objek `scaler` siap dipakai untuk transformasi data baru.  

- **find_first(df, keys)**  
  → Mencari kolom pertama dalam DataFrame yang cocok dengan daftar kata kunci (`keys`).  
  - Hanya mengembalikan kolom numerik.  
  - Output: nama kolom atau `None` jika tidak ditemukan.  

- **add_time_features(df, time_col)**  
  → Menambahkan fitur berbasis waktu dari kolom datetime.  
  - Fitur yang ditambahkan: `hour`, `dayofweek`, `month`.  
  - Encoding siklikal: `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `mon_sin`, `mon_cos`.  
  - Output: DataFrame dengan kolom tambahan.  

- **add_berawan_features(df)**  
  → Menambahkan fitur turunan untuk mendeteksi pola cuaca berawan.  
  - **Kelembapan (RH):** `rh_diff3`, `rh_diff6`.  
  - **Suhu:** `temp_diff3`, `temp_diff6`.  
  - **Tekanan:** `press_diff3`, `press_diff6`.  
  - **Radiasi:** `rad_std3`, `rad_std6` (rolling std).  
  - Output: DataFrame dengan fitur tambahan.  

---

Bagian ini menyiapkan fungsi-fungsi penting untuk preprocessing data cuaca:  
- **Windowing** untuk data time series.  
- **Scaler loader** agar preprocessing konsisten.  
- **Feature engineering** berbasis waktu & kondisi atmosfer.  


In [3]:
def make_windows(data, labels, seq_len, horizon):
    Xs, ys = [], []
    N = len(data)
    for i in range(N - seq_len - horizon + 1):
        Xs.append(data[i:i+seq_len])
        ys.append(labels[i+seq_len + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int32)

def load_scaler_from_json(path):
    with open(path, "r") as f:
        scaler_data = json.load(f)
    scaler = StandardScaler()
    scaler.mean_ = np.array(scaler_data["mean"])
    scaler.scale_ = np.array(scaler_data["std"])
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = len(scaler.mean_)
    return scaler

def find_first(df, keys):
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in keys):
            if pd.api.types.is_numeric_dtype(df[c]):
                return c
    return None

def add_time_features(df, time_col):
    dt = pd.to_datetime(df[time_col])
    df["hour"] = dt.dt.hour
    df["dayofweek"] = dt.dt.dayofweek
    df["month"] = dt.dt.month
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["dow_sin"]  = np.sin(2*np.pi*df["dayofweek"]/7)
    df["dow_cos"]  = np.cos(2*np.pi*df["dayofweek"]/7)
    df["mon_sin"]  = np.sin(2*np.pi*df["month"]/12)
    df["mon_cos"]  = np.cos(2*np.pi*df["month"]/12)
    return df

def add_berawan_features(df):
    rh_col = find_first(df, ["rh", "humid", "kelembap", "kelembaban"])
    if rh_col:
        df["rh_diff3"] = df[rh_col] - df[rh_col].shift(3)
        df["rh_diff6"] = df[rh_col] - df[rh_col].shift(6)
    temp_col = find_first(df, ["suhu", "temp"])
    if temp_col:
        df["temp_diff3"] = df[temp_col] - df[temp_col].shift(3)
        df["temp_diff6"] = df[temp_col] - df[temp_col].shift(6)
    press_col = find_first(df, ["tekan", "press"])
    if press_col:
        df["press_diff3"] = df[press_col] - df[press_col].shift(3)
        df["press_diff6"] = df[press_col] - df[press_col].shift(6)
    rad_col = find_first(df, ["radiasi", "solar", "sun"])
    if rad_col:
        df["rad_std3"] = df[rad_col].rolling(3, min_periods=1).std()
        df["rad_std6"] = df[rad_col].rolling(6, min_periods=1).std()
    return df

### Memuat Model Tersimpan

- **CNN Model**  
  `cnn_model = load_model("../Result Model/cnn_model.h5")`  
  → Memuat model Convolutional Neural Network (CNN) yang sudah dilatih sebelumnya.  
  → Digunakan untuk mengekstraksi fitur visual dari citra cuaca dan menghasilkan prediksi berbasis gambar.

- **Forecasting Model**  
  `forecast_model = load_model("../Result Model/forecasting_model.h5")`  
  → Memuat model forecasting (misalnya LSTM atau model sekuensial lain) yang sudah dilatih.  
  → Digunakan untuk menangkap pola temporal (trend, siklus, fluktuasi) dari data deret waktu cuaca.

---

Pada tahap ini, kedua model inti (CNN & Forecasting) sudah siap digunakan dalam pipeline prediksi cuaca.


In [6]:
cnn_model = load_model(r"../Result Model/cnn_model.h5")
forecast_model = load_model(r"../Result Model/forecasting_model.h5")

### Bagian 5: Memuat Scaler untuk Preprocessing

- **Load Scaler dari JSON**  
  `scaler = load_scaler_from_json("Hasil_Forecasting/scaler.json")`  
  → Memuat kembali objek **StandardScaler** yang sebelumnya disimpan dalam format JSON.  

- **Fungsi Utama**  
  - Menjamin preprocessing data baru **konsisten** dengan data training.  
  - Menggunakan parameter `mean` dan `std` yang sama seperti saat model dilatih.  
  - Menghindari perbedaan distribusi data antara training dan inferensi.  

- **Output**  
  Objek `scaler` siap digunakan untuk transformasi data input sebelum masuk ke model.


In [7]:
scaler = load_scaler_from_json("Hasil_Forecasting/scaler.json")

### Data Generator & Prediksi CNN

- **Direktori Data Uji**  
  - `TEST_DIR = r"../Dataset/Awan/test"`  
    → Menentukan path dataset uji (struktur folder per kelas).

- **ImageDataGenerator**  
  - `augmen_test = ImageDataGenerator(rescale=1./255)`  
    → Normalisasi pixel citra ke rentang [0,1].

- **Flow from Directory**  
  - `test_generator = augmen_test.flow_from_directory(...)`  
    → Membaca data uji dari folder dengan parameter:  
    - `target_size=(224,224)` → Resize citra ke ukuran input model.  
    - `batch_size=64` → Jumlah citra per batch.  
    - `class_mode='categorical'` → Label dalam bentuk one-hot encoding.  
    - `shuffle=False` → Urutan data tetap, penting untuk evaluasi.  
    - `seed=42` → Reproducibility.

- **Prediksi Model CNN**  
  - `probs_cnn = cnn_model.predict(test_generator, verbose=0)`  
    → Menghasilkan probabilitas prediksi untuk setiap kelas.  
  - `y_true_cnn = test_generator.classes`  
    → Label ground truth dari generator.

---

Bagian ini menyiapkan **pipeline evaluasi CNN**:  
- Data uji dibaca langsung dari folder.  
- Citra dinormalisasi & diresize sesuai input model.  
- Model menghasilkan **probabilitas prediksi** (`probs_cnn`).  
- Label asli (`y_true_cnn`) disimpan untuk evaluasi performa (confusion matrix, classification report, dsb).


In [8]:
TEST_DIR = r"../Dataset/Awan/test"
augmen_test = ImageDataGenerator(rescale=1./255)

test_generator = augmen_test.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

probs_cnn = cnn_model.predict(test_generator, verbose=0)
y_true_cnn = test_generator.classes

Found 253 images belonging to 3 classes.


### Preprocessing & Forecasting Pipeline

- **Load Dataset & Normalisasi Kolom**  
  - `pd.read_csv(r"../Dataset/Suhu/Prakiraan/dataset_cuaca_manokwari.csv")`  
    → Membaca dataset prakiraan cuaca Manokwari.  
  - `[c.strip().lower() for c in df_test.columns]`  
    → Normalisasi nama kolom (lowercase, hapus spasi).

- **Feature Engineering**  
  - `add_time_features(df_test, time_col="datetime")`  
    → Menambahkan fitur berbasis waktu (jam, hari, bulan, dsb).  
  - `add_berawan_features(df_test)`  
    → Menambahkan indikator kondisi berawan.

- **Handling Missing Values**  
  - `df_test = df_test.bfill().ffill()`  
    → Mengisi nilai kosong dengan backward-fill & forward-fill.

- **Encoding Target**  
  - `LabelEncoder().fit_transform(df_test[TARGET_COLUMN])`  
    → Mengubah label target menjadi integer.  
  - Simpan `le.classes_` untuk inverse transform hasil prediksi.

- **Feature Selection**  
  - `feature_cols = [c for c in df_test.columns if ...]`  
    → Eksklusi `TARGET_COLUMN` & `datetime`.  
    → Hanya ambil kolom numerik.

- **Scaling Features**  
  - `scaler.n_features_in_` → Jumlah fitur yang diharapkan scaler.  
  - `len(feature_cols)` → Jumlah fitur numerik terpilih.  
  - `scaler.transform(df_test[feature_cols])` → Normalisasi fitur numerik.  
  - Hasil check: **18 vs 18** → konsisten.

- **Windowing Data**  
  - `make_windows(X, y, SEQ_LEN, HORIZON)`  
    → Membuat sequence input (`X_ts`) & target (`y_true_fc`).  
    → `SEQ_LEN`: panjang input sequence.  
    → `HORIZON`: panjang horizon prediksi.

- **Forecasting**  
  - `forecast_model.predict([X_ts, X_ts], verbose=0)`  
    → Model menerima dual-input `[X_ts, X_ts]`.  
    → Output: `(N_ts, 8)` → probabilitas prediksi untuk horizon 8.

---

Bagian ini menyusun **pipeline forecasting**:  
- Dataset dibaca & dinormalisasi.  
- Fitur waktu & berawan ditambahkan.  
- Missing values di-handle.  
- Target di-encode.  
- Fitur numerik dipilih & diskalakan.  
- Data diubah menjadi window time-series.  
- Model menghasilkan probabilitas prediksi untuk horizon 8.


In [ ]:
df_test = pd.read_csv(r"../Dataset/Suhu/Prakiraan/dataset_cuaca_manokwari.csv")
df_test.columns = [c.strip().lower() for c in df_test.columns]

df_test = add_time_features(df_test, time_col="datetime")
df_test = add_berawan_features(df_test)

df_test = df_test.bfill().ffill()

le = LabelEncoder()
df_test[TARGET_COLUMN] = le.fit_transform(df_test[TARGET_COLUMN])

feature_cols = [
    c for c in df_test.columns
    if c not in [TARGET_COLUMN, "datetime"] and pd.api.types.is_numeric_dtype(df_test[c])
]

print("Scaler expects:", scaler.n_features_in_)
print("Feature cols:", len(feature_cols))

df_test[feature_cols] = scaler.transform(df_test[feature_cols])

X_ts, y_true_fc = make_windows(
    df_test[feature_cols].values,
    df_test[TARGET_COLUMN].values,
    SEQ_LEN,
    HORIZON
)

probs_fc = forecast_model.predict([X_ts, X_ts], verbose=0)  # (N_ts, 8)

Scaler expects: 18
Feature cols: 18


c:\Users\ASUS\anaconda3\envs\ml-env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


### Reduksi Kelas Prediksi (8 → 3)

- **Mapping Kelas**  
  - `mapping = {0:1, 1:0, 2:0, 3:2, 4:2, 5:2, 6:2, 7:1}`  
    → Mendefinisikan pemetaan dari 8 kelas detail ke 3 kelas utama:  
    - `0` → Cerah  
    - `1` → Berawan  
    - `2` → Hujan  

- **Agregasi Probabilitas**  
  - `probs_fc_3 = np.zeros((probs_fc.shape[0], 3))`  
    → Menyediakan array kosong untuk menampung probabilitas 3 kelas baru.  
  - `for old_idx, new_idx in mapping.items(): probs_fc_3[:, new_idx] += probs_fc[:, old_idx]`  
    → Menjumlahkan probabilitas kelas lama sesuai mapping.

- **Normalisasi Ulang**  
  - `probs_fc_3 = probs_fc_3 / probs_fc_3.sum(axis=1, keepdims=True)`  
    → Menormalkan kembali agar setiap baris probabilitas berjumlah 1.

---

Bagian ini menyederhanakan output model:  
- Dari **8 kelas detail** → menjadi **3 kategori utama** (Cerah, Berawan, Hujan).  
- Probabilitas kelas lama digabung sesuai mapping.  
- Hasil akhir (`probs_fc_3`) siap dipakai untuk evaluasi atau prediksi final.

In [11]:
mapping = {
    0: 1,  # Berawan -> Berawan
    1: 0,  # Cerah -> Cerah
    2: 0,  # Cerah Berawan -> Cerah
    3: 2,  # Hujan Petir -> Hujan
    4: 2,  # Hujan Ringan -> Hujan
    5: 2,  # Hujan Sedang -> Hujan
    6: 2,  # Petir -> Hujan
    7: 1   # Udara Kabur -> Berawan
}

probs_fc_3 = np.zeros((probs_fc.shape[0], 3))
for old_idx, new_idx in mapping.items():
    probs_fc_3[:, new_idx] += probs_fc[:, old_idx]

probs_fc_3 = probs_fc_3 / probs_fc_3.sum(axis=1, keepdims=True)


### Ensemble Evaluasi (CNN + Forecasting)

- **Sinkronisasi Panjang Data**  
  - `N = min(len(probs_cnn), len(probs_fc_3))`  
    → Menyamakan jumlah sampel antara output CNN dan Forecasting.  
  - `probs_cnn, probs_fc_3 = probs_cnn[:N], probs_fc_3[:N]`  
    → Memotong agar keduanya punya panjang sama.  
  - `y_true = y_true_cnn[:N]`  
    → Label ground truth disesuaikan dengan jumlah sampel.

- **Ensemble Probabilitas**  
  - `probs_ens = (probs_cnn + probs_fc_3) / 2.0`  
    → Menggabungkan prediksi CNN & Forecasting dengan rata-rata sederhana.

- **Prediksi Final**  
  - `y_pred = np.argmax(probs_ens, axis=1)`  
    → Mengambil kelas dengan probabilitas tertinggi sebagai prediksi akhir.

- **Evaluasi Model**  
  - `classification_report(y_true, y_pred, target_names=["Cerah","Berawan","Hujan"], digits=3)`  
    → Menampilkan precision, recall, f1-score per kelas.  
  - `confusion_matrix(y_true, y_pred)`  
    → Menampilkan matriks kebingungan untuk analisis kesalahan.

---

Bagian ini melakukan **ensemble sederhana** antara model CNN dan Forecasting:  
- Probabilitas digabung dengan rata-rata.  
- Prediksi akhir diambil dari probabilitas tertinggi.  
- Evaluasi dilakukan dengan **classification report** & **confusion matrix** untuk 3 kelas utama (Cerah, Berawan, Hujan).

### Hasil Evaluasi Ensemble

- **Classification Report**  
  - **Cerah** → precision 0.750, recall 0.492, f1-score 0.594 (support: 61)  
  - **Berawan** → precision 0.826, recall 0.942, f1-score 0.880 (support: 156)  
  - **Hujan** → precision 0.971, recall 0.944, f1-score 0.958 (support: 36)  
  - **Accuracy keseluruhan** → 0.834 (253 sampel)  
  - **Macro avg** → precision 0.849, recall 0.793, f1-score 0.811  
  - **Weighted avg** → precision 0.828, recall 0.834, f1-score 0.822  

- **Confusion Matrix**  
  - [[ 30 30 1] → Cerah diprediksi benar 30x, salah ke Berawan 30x, salah ke Hujan 1x 
  - [ 9 147 0] → Berawan diprediksi benar 147x, salah ke Cerah 9x
  - [ 1 1 34]] → Hujan diprediksi benar 34x, salah ke Cerah 1x, salah ke Berawan 1x

---

Interpretasi:  
- **Berawan** dan **Hujan** terklasifikasi sangat baik (recall > 0.94).  
- **Cerah** masih sering tertukar dengan **Berawan** (recall hanya 0.492).  
- Secara keseluruhan, **akurasi ensemble = 83.4%**, dengan f1-score tertinggi pada kelas **Hujan** (0.958).  
- Model lebih “aman” dalam mendeteksi kondisi hujan, tapi masih perlu perbaikan untuk membedakan **Cerah vs Berawan**.


In [12]:
N = min(len(probs_cnn), len(probs_fc_3))
probs_cnn, probs_fc_3 = probs_cnn[:N], probs_fc_3[:N]
y_true = y_true_cnn[:N]

probs_ens = (probs_cnn + probs_fc_3) / 2.0
y_pred = np.argmax(probs_ens, axis=1)

print(classification_report(y_true, y_pred, target_names=["Cerah","Berawan","Hujan"], digits=3))
print(confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

       Cerah      0.750     0.492     0.594        61
     Berawan      0.826     0.942     0.880       156
       Hujan      0.971     0.944     0.958        36

    accuracy                          0.834       253
   macro avg      0.849     0.793     0.811       253
weighted avg      0.828     0.834     0.822       253

[[ 30  30   1]
 [  9 147   0]
 [  1   1  34]]


### Membangun & Menyimpan Model Ensemble

- **Freeze Model Dasar**  
  - `cnn_model.trainable = False`  
  - `forecast_model.trainable = False`  
    → Membekukan bobot CNN & Forecasting agar tidak ikut dilatih ulang.

- **Input Layer**  
  - `img_input = Input(shape=cnn_model.input_shape[1:], name="img_input")`  
    → Input citra untuk CNN.  
  - `ts_input1 = Input(shape=forecast_model.input_shape[0][1:], name="ts_input1")`  
  - `ts_input2 = Input(shape=forecast_model.input_shape[1][1:], name="ts_input2")`  
    → Input time-series untuk Forecasting.

- **Output Probabilitas Model Dasar**  
  - `cnn_probs = cnn_model(img_input)` → Probabilitas CNN (3 kelas).  
  - `fc_probs_8 = forecast_model([ts_input1, ts_input2])` → Probabilitas Forecasting (8 kelas).

- **Mapping 8 → 3 Kelas**  
  - `M = np.zeros((8, 3))` + loop mapping → Matriks konversi kelas.  
  - `fc_logits_3 = Lambda(lambda x: tf.matmul(x, M_const), name="map8to3")(fc_probs_8)`  
    → Mengubah output 8 kelas menjadi 3 kelas.  
  - `fc_probs_3 = Lambda(lambda x: x / tf.reduce_sum(x, axis=-1, keepdims=True), name="normalize3")(fc_logits_3)`  
    → Normalisasi ulang agar probabilitas valid.

- **Ensemble Layer**  
  - `ens_probs = Average(name="ensemble_avg")([cnn_probs, fc_probs_3])`  
    → Menggabungkan prediksi CNN & Forecasting dengan rata-rata.

- **Bangun Model Ensemble**  
  - `ensemble_model = Model(inputs=[img_input, ts_input1, ts_input2], outputs=ens_probs, name="cnn_forecast_ensemble")`  
    → Membuat model gabungan dengan 3 input & 1 output (3 kelas).

- **Compile & Save**  
  - `ensemble_model.compile(optimizer="adam", loss="categorical_crossentropy")`  
    → Menentukan optimizer & loss function.  
  - `ensemble_model.save("../Result Model/cnn_forecast_ensemble_savedmodel", save_format="tf")`  
    → Menyimpan model ensemble dalam format TensorFlow SavedModel.

---

Bagian ini membangun **arsitektur ensemble end-to-end**:  
- CNN (citra) + Forecasting (time-series) digabung.  
- Output Forecasting (8 kelas) dipetakan ke 3 kelas utama.  
- Probabilitas CNN & Forecasting dirata-rata.  
- Model akhir disimpan sebagai **SavedModel** untuk deployment.


In [ ]:
cnn_model.trainable = False
forecast_model.trainable = False

img_input = Input(shape=cnn_model.input_shape[1:], name="img_input")
ts_input1 = Input(shape=forecast_model.input_shape[0][1:], name="ts_input1")
ts_input2 = Input(shape=forecast_model.input_shape[1][1:], name="ts_input2")

cnn_probs = cnn_model(img_input)                  # (None,3)
fc_probs_8 = forecast_model([ts_input1, ts_input2])  # (None,8)

M = np.zeros((8, 3), dtype=np.float32)
for old_idx, new_idx in mapping.items():
    M[old_idx, new_idx] = 1.0
M_const = tf.constant(M, dtype=tf.float32)

fc_logits_3 = Lambda(lambda x: tf.matmul(x, M_const), name="map8to3")(fc_probs_8)
fc_probs_3 = Lambda(lambda x: x / tf.reduce_sum(x, axis=-1, keepdims=True),
                    name="normalize3")(fc_logits_3)

ens_probs = Average(name="ensemble_avg")([cnn_probs, fc_probs_3])

ensemble_model = Model(inputs=[img_input, ts_input1, ts_input2],
                       outputs=ens_probs,
                       name="cnn_forecast_ensemble")

ensemble_model.compile(optimizer="adam", loss="categorical_crossentropy")

ensemble_model.save(r"../Result Model/cnn_forecast_ensemble_savedmodel", save_format="tf")
print("Save model complete.")


INFO:tensorflow:Assets written to: ../Result Model/cnn_forecast_ensemble_savedmodel\assets


INFO:tensorflow:Assets written to: ../Result Model/cnn_forecast_ensemble_savedmodel\assets


Save model complete.


### Penyimpanan & Kompresi Model Ensemble

- **Direktori Penyimpanan**  
  - `save_dir = r"../Result Model/cnn_forecast_ensemble_savedmodel"`  
    → Menentukan lokasi penyimpanan model ensemble.

- **Simpan Model**  
  - `ensemble_model.save(save_dir, save_format="tf")`  
    → Menyimpan model dalam format **TensorFlow SavedModel**.

- **Persiapan File ZIP**  
  - `zip_filename = save_dir + ".zip"`  
    → Menentukan nama file ZIP hasil kompresi.  
  - `if os.path.exists(zip_filename): os.remove(zip_filename)`  
    → Menghapus file ZIP lama jika sudah ada.

- **Kompresi Model**  
  - `shutil.make_archive(save_dir, 'zip', save_dir)`  
    → Membuat arsip ZIP dari folder model.

- **Konfirmasi**  
  - `print(f"Model berhasil disimpan dan di-zip: {zip_filename}")`  
    → Menampilkan pesan sukses setelah proses selesai.

---

Bagian ini memastikan model ensemble:  
- Disimpan dalam format **SavedModel**.  
- Dikompresi menjadi **ZIP** agar mudah dipindahkan atau dibagikan.  
- File lama otomatis dihapus untuk mencegah konflik.


In [ ]:


save_dir = r"../Result Model/cnn_forecast_ensemble_savedmodel"
ensemble_model.save(save_dir, save_format="tf")

zip_filename = save_dir + ".zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)  # hapus zip lama kalau ada

shutil.make_archive(save_dir, 'zip', save_dir)

print(f"Model berhasil disimpan dan di-zip: {zip_filename}")


INFO:tensorflow:Assets written to: ../Result Model/cnn_forecast_ensemble_savedmodel\assets


INFO:tensorflow:Assets written to: ../Result Model/cnn_forecast_ensemble_savedmodel\assets


Model berhasil disimpan dan di-zip: ../Result Model/cnn_forecast_ensemble_savedmodel.zip
